# Submesoscale currents: Colab quick start

This notebook runs the pipeline on a free Colab GPU (**Runtime → Change runtime type → T4 GPU**).

1. **Demo:** a synthetic SQG basin; no account needed; takes about 5 minutes.
2. **Real data:** the Western Mediterranean from Copernicus Marine. This needs a free account.

If the GitHub repository is private, first create a *fine-grained personal access token* with read access, then use
`https://<token>@github.com/...` in the clone command below.

In [ ]:
REPO = "https://github.com/divyagoyal6224/submesoscale-currents.git"
!git clone -q $REPO submesoscale 2>/dev/null || (cd submesoscale && git pull -q)
%cd submesoscale
!pip install -q -e ".[data]"
import torch; print("GPU:", torch.cuda.is_available())

## 1. Synthetic demo (all stages)

In [ ]:
!submeso all -c configs/demo_synthetic.yaml

In [ ]:
import json
from IPython.display import Image, Video, display
m = json.load(open("runs/demo_synthetic/metrics_osse.json"))
for tag in ("lr", "sr"):
    print(tag, {k: round(v, 2) for k, v in m[tag].items()})
for f in ("osse_snapshot", "osse_spectra", "drifter_scatter", "training_history"):
    display(Image(f"runs/demo_synthetic/figures/{f}.png"))

In [ ]:
Video("runs/demo_synthetic/figures/animation_sr.mp4", embed=True, width=720)

## 2. Real data: Western Mediterranean

Your Copernicus Marine credentials are read with `getpass`, so they never appear in the notebook.
Downloads are saved in yearly files and resume if the run is interrupted. To keep the data between
sessions, mount Google Drive and add `-o data.root=/content/drive/MyDrive/submeso/wmed`.

In [ ]:
import os, getpass
os.environ["COPERNICUSMARINE_SERVICE_USERNAME"] = input("Copernicus username: ")
os.environ["COPERNICUSMARINE_SERVICE_PASSWORD"] = getpass.getpass("Copernicus password: ")

In [ ]:
CFG = "configs/wmed.yaml"
# Colab tip: shorten the training years for a first run, e.g. add
#   -o "period.train=[2017-01-01, 2019-12-31]"
!submeso download -c $CFG
!submeso prepare  -c $CFG
!submeso pairs    -c $CFG

In [ ]:
!submeso train -c $CFG -o train.num_workers=2

In [ ]:
!submeso evaluate    -c $CFG
!submeso reconstruct -c $CFG
!submeso validate    -c $CFG
!submeso visualize   -c $CFG

In [ ]:
print(open("runs/wmed_sr/metrics_validation.json").read())
display(Image("runs/wmed_sr/figures/drifter_scatter.png"))
display(Image("runs/wmed_sr/figures/real_spectra.png"))